# ResNet Blocks: Why Skip Connections Unlock Depth

Companion notebook for the [CNNs and Residual Learning](https://codemia.io/courses/deep_learning_foundations/cnns_and_residual_learning) lesson on Codemia.

**What you will build**: a ResNet block implemented from scratch in PyTorch, trained on CIFAR-10 with and without the skip connection, with convergence curves that make the point visually.

**Time**: ~60 minutes on a T4 GPU  
**GPU required**: strongly recommended (this notebook trains convnets on images). It will run on CPU but each epoch will take ~5 minutes.

## Setup

In [ ]:
%pip install -q torch torchvision matplotlib

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')

## Concept recap

Before ResNet (2015), networks deeper than ~20 layers got *worse* on the training set, not just the validation set. The problem was not overfitting — it was an optimization failure. The ResNet paper's fix was embarrassingly simple:

$$y = F(x) + x$$

Each block learns a *residual* $F(x)$ added to its input. If $F$ is tiny, the block is an identity — so a 100-layer network can always degenerate into a shallower one. More importantly, the gradient of the loss with respect to early layers now has an extra `+1` term (the derivative of the identity branch), so it cannot vanish purely from depth.

We will build a basic ResNet block and train two networks on CIFAR-10: with and without the skip.

## Implementation

### Step 1 — The ResNet basic block

Two 3x3 convs, BatchNorm after each, ReLU in between. The skip wires the input to the output before the final ReLU. We make the skip toggleable so we can ablate it.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_ch, out_ch, stride=1, use_skip=True):
        super().__init__()
        self.use_skip = use_skip
        # First conv can downsample (stride>1). Padding=1 keeps spatial size when stride=1.
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        # Second conv always stride 1. BN absorbs the conv bias, so bias=False.
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        # Shortcut — may need a 1x1 conv to match shape when stride or channels change.
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_skip:
            out = out + self.shortcut(x)
        return F.relu(out)

### Step 2 — A deep network stacked from basic blocks

Pattern: stem conv -> stage of N blocks at 64 channels -> stage at 128 (stride 2 at entry) -> stage at 256 (stride 2) -> global avg pool -> linear. With `blocks_per_stage=3` we end up with ~20 conv layers, enough depth for the skip connection to matter but still fast on a T4.

In [ ]:
class TinyResNet(nn.Module):
    def __init__(self, blocks_per_stage=3, n_classes=10, use_skip=True):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.stage1 = self._make_stage(64, 64, blocks_per_stage, stride=1, use_skip=use_skip)
        self.stage2 = self._make_stage(64, 128, blocks_per_stage, stride=2, use_skip=use_skip)
        self.stage3 = self._make_stage(128, 256, blocks_per_stage, stride=2, use_skip=use_skip)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, n_classes)
    def _make_stage(self, in_ch, out_ch, n, stride, use_skip):
        # First block can change shape; the rest are shape-preserving.
        layers = [BasicBlock(in_ch, out_ch, stride=stride, use_skip=use_skip)]
        for _ in range(n - 1):
            layers.append(BasicBlock(out_ch, out_ch, stride=1, use_skip=use_skip))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.stage2(x); x = self.stage3(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

m = TinyResNet()
print(m)
print('params:', sum(p.numel() for p in m.parameters()))

### Step 3 — CIFAR-10 data

Standard CIFAR-10 with cheap augmentation (random crop + horizontal flip) to keep val loss honest. Batch size 256 fits on a T4.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),  # crop augmentation is a cheap but effective regularizer
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

train_ds = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=train_tf)
test_ds  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_tf)

# num_workers=2 avoids hogging all CPU on Colab.
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
print('train batches:', len(train_loader), '  test batches:', len(test_loader))

### Step 4 — Training loop

In [ ]:
def train_one(model, epochs=8, lr=0.1):
    model = model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    # Cosine decay fits naturally inside one short run — stable convergence, no tuning needed.
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        model.train()
        t0 = time.time()
        losses = []
        for X, y in train_loader:
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad()
            loss = F.cross_entropy(model(X), y)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        sched.step()
        train_loss = sum(losses) / len(losses)

        # Eval pass — BN switches to running stats in eval() mode.
        model.eval()
        v_losses, correct, total = [], 0, 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                logits = model(X)
                v_losses.append(F.cross_entropy(logits, y).item())
                correct += (logits.argmax(-1) == y).sum().item()
                total += y.numel()
        history['train_loss'].append(train_loss)
        history['val_loss'].append(sum(v_losses) / len(v_losses))
        history['val_acc'].append(correct / total)
        dt = time.time() - t0
        print(f'  epoch {epoch+1:2d}/{epochs}  train={train_loss:.3f}  val={history["val_loss"][-1]:.3f}  acc={history["val_acc"][-1]:.3f}  ({dt:.1f}s)')
    return history

## Experiment: with vs. without skip

Same architecture, same data, same optimizer, same init seed. The only difference is whether `+ self.shortcut(x)` fires.

In [ ]:
EPOCHS = 8  # enough for the gap to be visible; bump to 30+ for serious accuracy

print('=== With skip connections ===')
torch.manual_seed(0)
hist_skip = train_one(TinyResNet(blocks_per_stage=3, use_skip=True), epochs=EPOCHS)

print('\n=== Without skip connections ===')
torch.manual_seed(0)
hist_noskip = train_one(TinyResNet(blocks_per_stage=3, use_skip=False), epochs=EPOCHS)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(hist_skip['train_loss'], label='skip, train')
ax[0].plot(hist_skip['val_loss'], '--', label='skip, val')
ax[0].plot(hist_noskip['train_loss'], label='no skip, train')
ax[0].plot(hist_noskip['val_loss'], '--', label='no skip, val')
ax[0].set_title('loss'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(hist_skip['val_acc'], label='skip')
ax[1].plot(hist_noskip['val_acc'], label='no skip')
ax[1].set_title('val accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'final val acc  with skip : {hist_skip["val_acc"][-1]:.3f}')
print(f'final val acc  no skip   : {hist_noskip["val_acc"][-1]:.3f}')

## Variations

### Scaling depth

The real ResNet claim is about **depth**: adding the skip lets you make the network deeper without it getting worse. Rerun with more blocks per stage and compare the gap.

In [ ]:
# WARNING: this doubles training time. Skip on CPU.
DEEP = 5   # 5 blocks per stage = ~32 conv layers
EPOCHS_DEEP = 6

print(f'=== deep + skip (blocks_per_stage={DEEP}) ===')
torch.manual_seed(0)
hist_deep_skip = train_one(TinyResNet(blocks_per_stage=DEEP, use_skip=True), epochs=EPOCHS_DEEP)

print(f'\n=== deep + NO skip ===')
torch.manual_seed(0)
hist_deep_noskip = train_one(TinyResNet(blocks_per_stage=DEEP, use_skip=False), epochs=EPOCHS_DEEP)

print()
print(f'deep + skip   final acc: {hist_deep_skip["val_acc"][-1]:.3f}')
print(f'deep no skip  final acc: {hist_deep_noskip["val_acc"][-1]:.3f}')

## What to try next

- Add a **BottleneckBlock** (1x1 -> 3x3 -> 1x1 convs). This is how ResNet-50/101/152 stay cheap at high depth.
- Replace BatchNorm with GroupNorm and see what changes at small batch sizes.
- Initialize the final BN in each residual branch to zero ("Fixup" / "ZeroInit") — a neat trick that lets you train ResNets *without* the residual being active at the start.
- Scale to 50 epochs with stronger augmentation (MixUp, CutMix) to approach 92% on CIFAR-10.